# 🧠 Autonomous Multi-Agent Generative AI Decision Intelligence Platform

This notebook orchestrates a no-retraining AI analysis pipeline:

1. **Install & Setup** — Dependencies + Ollama + Pull base model
2. **Load Pre-Tuned Model From Hugging Face** — Download trained artifacts
3. **Load Model into Ollama** — Create business-analyst model + initialize LLM client
4. **Load/Generate Datasets** — Prepare business & tech data
5. **Data Exploration** — EDA with pandas & visualizations
6. **Auto-Detect Dataset Types** — Dynamic schema classification
7. **Run Individual Agent Analysis** — Each AI agent analyzes its domain
8. **Run Multi-Agent Orchestration** — Collaborative strategic synthesis
9. **Launch Streamlit Dashboard** — Interactive web UI

---

**Hardware:** 32GB RAM + RTX A6000 (48GB VRAM)  
**LLM Runtime:** Ollama (`business-analyst`)  
**Model Source:** Hugging Face (pre-tuned artifacts)  
**Framework:** Multi-agent system with 5 specialized AI agents

## 1️⃣ Install Dependencies & Setup Ollama

In [ ]:
# ── Step 1a: Install Ollama daemon (Linux) ──
!curl -fsSL https://ollama.com/install.sh | sh

# ── Step 1b: Start Ollama server in background ──
import subprocess
import time

print("Starting Ollama server in the background...")
process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)  # Wait for server to start
print("✅ Ollama server started!")

In [ ]:
# ── Step 1c: Install Python dependencies ──
%pip install -r requirements.txt

# Verify imports
import streamlit
import pandas as pd
import numpy as np
import plotly
import sklearn
import ollama

print("✅ All dependencies installed!")
print(f"  📊 pandas {pd.__version__}")
print(f"  🔢 numpy {np.__version__}")
print(f"  📈 plotly {plotly.__version__}")
print(f"  🧪 scikit-learn {sklearn.__version__}")
print(f"  🖥️ streamlit {streamlit.__version__}")

In [ ]:
# ── Step 1d: Pull the base Phi-3 Mini model ──
# This is used as the base for fine-tuning

import ollama

BASE_MODEL = "phi3:mini"

print(f"📥 Pulling base model '{BASE_MODEL}'...")
print("   (This may take a few minutes on first run)")

try:
    ollama.pull(BASE_MODEL)
    print(f"✅ Base model '{BASE_MODEL}' is ready!")
    
    # Verify it's available
    models_response = ollama.list()
    if hasattr(models_response, 'models'):
        model_list = models_response.models
    elif isinstance(models_response, dict):
        model_list = models_response.get('models', [])
    else:
        model_list = []
    
    available = []
    for m in model_list:
        if hasattr(m, 'model'):
            available.append(m.model)
        elif isinstance(m, dict):
            available.append(m.get('model', m.get('name', str(m))))
        else:
            available.append(str(m))
    print(f"   Available models: {available}")
except Exception as e:
    print(f"⚠️ Could not pull model: {e}")
    print("   The system will use fallback responses.")

---

## 2️⃣ Load Pre-Tuned Model Artifacts From Hugging Face

Skip fine-tuning entirely by downloading the published artifacts from Hugging Face.

Published repos:
- `HeshamXOR/business-analyst-phi3-mini-merged`
- `HeshamXOR/business-analyst-phi3-mini-lora`

In [1]:
# -- Step 2a: Download pre-tuned model artifacts from Hugging Face --
import os
import sys

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from finetune.hf_hub import download_model_from_hub

MERGED_REPO = "HeshamXOR/business-analyst-phi3-mini-merged"
LORA_REPO = "HeshamXOR/business-analyst-phi3-mini-lora"

hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")

print("📥 Downloading pre-tuned model artifacts from Hugging Face...")
if hf_token:
    print("   Using token from environment.")
else:
    print("   No token found in env. Public download mode.")

merged_local = "finetune/output/merged_model"
lora_local = "finetune/output/lora_adapter"

download_model_from_hub(repo_id=MERGED_REPO, local_dir=merged_local, token=hf_token)
print(f"✅ Downloaded merged model to: {merged_local}")

download_model_from_hub(repo_id=LORA_REPO, local_dir=lora_local, token=hf_token)
print(f"✅ Downloaded LoRA adapter to: {lora_local}")

print("\n✅ Hugging Face artifacts are ready. Skipping all training steps.")

📥 Downloading pre-tuned model artifacts from Hugging Face...
   No token found in env. Public download mode.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

✅ Downloaded LoRA adapter to: finetune/output/lora_adapter

✅ Hugging Face artifacts are ready. Skipping all training steps.


---

## 3️⃣ Load Fine-Tuned Model into Ollama

Create a custom Ollama model from the fine-tuned weights so all agents use it.

In [2]:
# ── Step 3a: Create Ollama Modelfile ──
from finetune.create_modelfile import create_modelfile

# Check if GGUF from full training exists, otherwise use prompt-tuning approach
import os
merged_path = "finetune/output/merged_model"

if os.path.exists(merged_path):
    print("✅ Fine-tuned model found! Creating Modelfile from trained weights...")
    print("")
    print("💡 To convert to GGUF (optional, for smaller model file):")
    print(f"   python llama.cpp/convert_hf_to_gguf.py {merged_path} --outtype q4_k_m")
    print("")

# Create Modelfile with optimized system prompt
modelfile_path = create_modelfile(
    base_model="phi3:mini",
    output_path="finetune/Modelfile"
)
print(f"\n📝 Modelfile created at: {modelfile_path}")

✅ Fine-tuned model found! Creating Modelfile from trained weights...

💡 To convert to GGUF (optional, for smaller model file):
   python llama.cpp/convert_hf_to_gguf.py finetune/output/merged_model --outtype q4_k_m

✅ Modelfile created: finetune/Modelfile

To create the Ollama model, run:
  ollama create business-analyst -f finetune/Modelfile

Then update .env:
  OLLAMA_MODEL=business-analyst

📝 Modelfile created at: finetune/Modelfile


In [3]:
# -- Step 3b: Register model with Ollama --
import os
import subprocess
import time
import ollama

ollama_host = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434")
client = ollama.Client(host=ollama_host)


def ensure_ollama_running(timeout_seconds: int = 30) -> None:
    """Start Ollama server if needed and wait until it responds."""
    try:
        client.list()
        print(f"✅ Ollama server is running at {ollama_host}")
        return
    except Exception:
        print("⚠️ Ollama server not reachable. Starting 'ollama serve'...")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    start = time.time()
    while time.time() - start < timeout_seconds:
        try:
            client.list()
            print(f"✅ Ollama server started at {ollama_host}")
            return
        except Exception:
            time.sleep(1)

    raise RuntimeError(
        "Could not reach Ollama server after starting it. "
        "Run 'ollama serve' manually and try again."
    )


ensure_ollama_running()

print("\n🔧 Creating custom Ollama model 'business-analyst'...")
create_proc = subprocess.run(
    ["ollama", "create", "business-analyst", "-f", "finetune/Modelfile"],
    capture_output=True,
    text=True,
)
if create_proc.returncode != 0:
    raise RuntimeError(create_proc.stderr.strip() or create_proc.stdout.strip())

print("✅ Custom model 'business-analyst' is ready!")

# Quick test
response = client.chat(
    model="business-analyst",
    messages=[
        {
            "role": "user",
            "content": "Analyze: Revenue=$4.5M, Growth=12%, Top region=NA at 42%",
        }
    ],
)

print("\n🧪 Test response from fine-tuned model:")
print(response["message"]["content"][:500])

⚠️ Ollama server not reachable. Starting 'ollama serve'...
✅ Ollama server started at http://127.0.0.1:11434

🔧 Creating custom Ollama model 'business-analyst'...
✅ Custom model 'business-analyst' is ready!

🧪 Test response from fine-tuned model:
# Business Data Analysis Report

## Executive Summary
This report analyzes the current business performance based on provided datasets and calculates key financial metrics to identify trends, patterns, anomalies, correlations, as well as providing data-backed recommendations. The primary focus is revenue growth with an emphasis on regional contributions across North America (NA). 

## Revenue Analysis
The total annual revenue stands at $4.5M, reflecting a robust yearly income for the business. N


In [4]:
# ── Step 3c: Initialize LLM Client with fine-tuned model ──
from llm.llm_client import LLMClient

# Use the fine-tuned business-analyst model for ALL subsequent analysis
llm_client = LLMClient(model="business-analyst")

# Check connection
status = llm_client.check_connection()
print("🔌 LLM Connection Status:")
for k, v in status.items():
    print(f"  {k}: {v}")

# Quick test
print("\n🧪 Quick test:")
response = llm_client.generate("Say 'Hello! I am ready to analyze business data.' in one sentence.")
print(f"  LLM says: {response}")

🔌 LLM Connection Status:
  connected: True
  base_url: http://localhost:11434
  model: business-analyst
  model_available: True
  available_models: ['business-analyst:latest', 'phi3:mini']

🧪 Quick test:
  LLM says: **Greetings, Data Analyst Ready State Confirmed: Analysis Mode Engaged.**




---

## 4️⃣ Load / Generate Datasets

4 realistic synthetic datasets:
- 📊 **Sales** — 1000 transactions with revenue, products, regions
- 🎯 **Marketing** — 500 campaigns with ROI, channels, conversions
- 👥 **Customers** — 800 profiles with LTV, churn risk, segments
- 🐙 **GitHub** — 600 repositories with stars, languages, quality scores

In [5]:
# Generate datasets if they don't exist
import os
from data.generate_datasets import generate_all_datasets

if not all(os.path.exists(f"data/{f}") for f in 
           ["sales_data.csv", "marketing_data.csv", "customers_data.csv", "github_repos.csv"]):
    print("Generating datasets...")
    generate_all_datasets("data")
else:
    print("✅ Datasets already exist!")

# Load all datasets
from utils.data_loader import load_all_datasets
datasets = load_all_datasets("data")

print(f"\n📂 Loaded {len(datasets)} datasets:")
for name, df in datasets.items():
    print(f"  📊 {name}: {df.shape[0]} rows × {df.shape[1]} columns")

✅ Datasets already exist!

📂 Loaded 4 datasets:
  📊 sales: 1000 rows × 9 columns
  📊 marketing: 500 rows × 12 columns
  📊 customers: 800 rows × 11 columns
  📊 github: 600 rows × 12 columns


## 5️⃣ Data Exploration & Visualization

In [8]:
# Quick look at each dataset
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"📊 {name.upper()} DATASET")
    print(f"{'='*60}")
    display(df.head())
    print(f"\nShape: {df.shape}")
    print(f"Missing values: {df.isnull().sum().sum()}")


📊 SALES DATASET


,date,product,category,region,revenue,units_sold,cost,profit_margin,discount_applied
0,2023-01-01,Support Plan Premium,Services,North America,8230.30,2,4691.27,0.43,0.10
1,2023-01-01,SSD Storage 2TB,Hardware,Europe,2691.47,1,1803.28,0.33,0.00
2,2023-01-01,USB-C Hub,Electronics,Middle East,360.31,1,227.00,0.37,0.05
3,2023-01-02,Security Package,Software,Asia Pacific,3531.72,1,2224.98,0.37,0.00
4,2023-01-02,GPU Accelerator Card,Hardware,North America,3929.76,8,2947.32,0.25,0.00



Shape: (1000, 9)
Missing values: 0

📊 MARKETING DATASET


,date,campaign_type,channel,impressions,clicks,ctr,conversions,conversion_rate,spend,revenue_generated,roi,cpc
0,2023-01-02,Brand Awareness,Google Ads,3287,156,0.0477,12,0.0772,35029.51,4667.23,-86.68,224.55
1,2023-01-03,Content Marketing,Email,1899,60,0.0320,2,0.0338,7365.42,924.91,-87.44,122.76
2,2023-01-05,Seasonal Promo,TikTok,139510,2650,0.0190,120,0.0453,17947.24,9012.87,-49.78,6.77
3,2023-01-06,Retargeting,TikTok,2768,76,0.0275,4,0.0560,14666.47,618.93,-95.78,192.98
4,2023-01-08,Brand Awareness,Instagram,16745,370,0.0221,13,0.0356,21529.84,3533.99,-83.59,58.19



Shape: (500, 12)
Missing values: 0

📊 CUSTOMERS DATASET


,customer_id,segment,industry,lifetime_value,account_age_months,purchase_frequency,satisfaction_score,support_tickets,churn_risk,nps_score,engagement_score
0,CUST-01001,Startup,Finance,805.06,7,0.9,8.0,3,0.00,70,63.1
1,CUST-01002,Startup,Media,3948.41,3,0.8,7.5,1,0.29,71,84.7
2,CUST-01003,Enterprise,Retail,38269.88,78,1.7,8.6,5,0.13,100,43.6
3,CUST-01004,Small Business,Media,6789.02,67,4.6,8.5,2,0.53,72,15.0
4,CUST-01005,Small Business,Manufacturing,4333.93,41,6.0,5.8,2,0.32,76,36.9



Shape: (800, 11)
Missing values: 0

📊 GITHUB DATASET


,repo_name,language,topic,stars,forks,open_issues,contributors,last_updated,code_quality_score,license,has_ci_cd,has_documentation
0,fast-craft-io,Kotlin,database,41,4,0,7,2024-05-30,2.3,NaN,True,False
1,turbo-forge,JavaScript,security,963,219,13,1,2024-08-17,2.4,MIT,False,False
2,awesome-vault,Java,web-framework,6,2,0,54,2024-03-18,2.5,Apache-2.0,False,False
3,ultra-api,TypeScript,security,71,25,1,4,2024-06-03,1.5,GPL-3.0,True,True
4,auto-vault,TypeScript,testing,123,15,3,3,2024-07-17,2.0,Apache-2.0,False,False



Shape: (600, 12)
Missing values: 63


In [9]:
# Compute KPIs for each dataset
from utils.analysis import compute_kpis

dataset_types = {"sales": "sales", "marketing": "marketing", 
                 "customers": "customers", "github": "github"}

all_kpis = {}
for name, dtype in dataset_types.items():
    if name in datasets:
        kpis = compute_kpis(datasets[name], dtype)
        all_kpis[name] = kpis
        print(f"\n📈 {name.upper()} KPIs:")
        for k, v in kpis.items():
            print(f"  {k}: {v}")


📈 SALES KPIs:
  total_revenue: 3967082.55
  avg_revenue_per_transaction: 3967.08
  total_units_sold: 4452
  avg_profit_margin: 26.1
  total_transactions: 1000
  unique_products: 21
  top_region: North America
  top_category: Services
  avg_discount: 6.7
  revenue_std: 3573.03

📈 MARKETING KPIs:
  total_spend: 13143917.7
  total_revenue_generated: 6447669.8
  overall_roi: -50.95
  avg_ctr: 2.81
  avg_conversion_rate: 3.57
  total_conversions: 24303
  total_impressions: 24208155
  avg_cpc: 135.31
  best_channel: YouTube
  best_campaign_type: Brand Awareness

📈 CUSTOMERS KPIs:
  total_customers: 800
  avg_lifetime_value: 7855.19
  median_lifetime_value: 3577.33
  avg_satisfaction: 7.2
  avg_churn_risk: 17.0
  high_risk_customers: 2
  avg_nps: 70.8
  avg_engagement: 40.1
  top_segment: Enterprise
  avg_account_age_months: 24.8

📈 GITHUB KPIs:
  total_repos: 600
  total_stars: 658509
  avg_stars: 1097.5
  median_stars: 130
  total_forks: 166789
  avg_code_quality: 2.4
  top_language: Pytho

In [10]:
# Create visualizations
from components.charts import (
    revenue_trend_chart, top_products_chart,
    campaign_performance_chart, customer_segments_chart,
    github_stats_chart
)

if "sales" in datasets:
    print("📊 Sales Visualizations")
    revenue_trend_chart(datasets["sales"]).show()
    top_products_chart(datasets["sales"]).show()

if "marketing" in datasets:
    print("🎯 Marketing Visualizations")
    campaign_performance_chart(datasets["marketing"]).show()

if "customers" in datasets:
    print("👥 Customer Visualizations")
    customer_segments_chart(datasets["customers"]).show()

if "github" in datasets:
    print("🐙 GitHub Visualizations")
    github_stats_chart(datasets["github"]).show()

📊 Sales Visualizations


/home/ubuntu/components/charts.py:70: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df_copy.groupby(pd.Grouper(key=date_col, freq="M"))[value_col].sum().reset_index()


🎯 Marketing Visualizations


👥 Customer Visualizations


🐙 GitHub Visualizations


## 6️⃣ Auto-Detect Dataset Types

The `DatasetDetector` automatically classifies any CSV by analyzing column names and data types.  
It works with sales, marketing, customer, financial, HR, inventory, tech, or any generic dataset.

In [11]:
from utils.dataset_detector import DatasetDetector, detect_all_datasets

# Auto-detect all loaded dataset types
detectors = detect_all_datasets(datasets)

print("🔍 Auto-Detection Results:")
print("="*60)
for name, detector in detectors.items():
    summary = detector.get_detection_summary()
    print(f"\n📊 {name.upper()}:")
    print(f"   Detected type: {summary['detected_type'].title()}")
    print(f"   Confidence:    {summary['confidence']:.0%}")
    print(f"   Date columns:  {summary['date_columns']}")
    print(f"   Monetary cols: {summary['monetary_columns']}")
    print(f"   Primary metric: {summary['primary_metric']}")
    print(f"   Recommended charts: {summary['recommended_charts']}")

🔍 Auto-Detection Results:

📊 SALES:
   Detected type: Sales
   Confidence:    100%
   Date columns:  ['date']
   Monetary cols: ['revenue', 'cost', 'profit_margin']
   Primary metric: revenue
   Recommended charts: 6

📊 MARKETING:
   Detected type: Marketing
   Confidence:    100%
   Date columns:  ['date', 'spend']
   Monetary cols: ['revenue_generated', 'roi', 'cpc']
   Primary metric: revenue_generated
   Recommended charts: 6

📊 CUSTOMERS:
   Detected type: Customers
   Confidence:    100%
   Date columns:  ['lifetime_value', 'account_age_months']
   Monetary cols: []
   Primary metric: nps_score
   Recommended charts: 6

📊 GITHUB:
   Detected type: Tech
   Confidence:    100%
   Date columns:  ['last_updated']
   Monetary cols: []
   Primary metric: stars
   Recommended charts: 6


In [12]:
# Auto-generated charts for any dataset
from components.charts import auto_generate_charts

for name, detector in detectors.items():
    chart_specs = detector.get_chart_recommendations()
    auto_charts = auto_generate_charts(datasets[name], chart_specs)
    
    print(f"\n📊 Auto-Charts for {name.upper()} ({len(auto_charts)} charts):")
    for title, fig in auto_charts[:3]:  # Show first 3
        fig.show()


📊 Auto-Charts for SALES (6 charts):


/home/ubuntu/components/charts.py:417: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df_copy.groupby(pd.Grouper(key=x_col, freq="M"))[y_col].sum().reset_index()


/home/ubuntu/components/charts.py:417: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df_copy.groupby(pd.Grouper(key=x_col, freq="M"))[y_col].sum().reset_index()



📊 Auto-Charts for MARKETING (6 charts):



📊 Auto-Charts for CUSTOMERS (6 charts):


/home/ubuntu/components/charts.py:417: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df_copy.groupby(pd.Grouper(key=x_col, freq="M"))[y_col].sum().reset_index()


/home/ubuntu/components/charts.py:417: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df_copy.groupby(pd.Grouper(key=x_col, freq="M"))[y_col].sum().reset_index()



📊 Auto-Charts for GITHUB (6 charts):


## 7️⃣ Run Individual Agent Analysis

Each agent uses the **fine-tuned business-analyst model** for specialized analysis:
- 📊 **Sales Analyst** — Revenue trends, product performance
- 🎯 **Marketing Analyst** — Campaign ROI, channel effectiveness
- 👥 **Customer Analyst** — Segmentation, churn risk (+ ML clustering)
- 💻 **Tech Analyst** — GitHub trends, code quality (+ ML anomaly detection)

In [9]:
from agents.sales_agent import SalesAnalystAgent
from agents.marketing_agent import MarketingAnalystAgent
from agents.customer_agent import CustomerAnalystAgent
from agents.tech_agent import TechAnalystAgent

# Sales Agent
print("📊 Running Sales Analyst Agent...")
sales_agent = SalesAnalystAgent(llm_client)
sales_report = sales_agent.analyze(datasets["sales"], include_trends=True)
print("✅ Sales analysis complete!")
print(f"⏱️ Time: {sales_agent.get_metadata()['execution_time_seconds']}s\n")
print(sales_report[:500] + "...")

📊 Running Sales Analyst Agent...


/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()
/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()


✅ Sales analysis complete!
⏱️ Time: 17.34s

# Sales Performance Analysis Report for Q1 - North America Region Focus

## Revenue Performance Overview
- Total revenue: $4.0M (flat growth)
- Avg revenue per transaction remains consistent at approximately $4,000
- Highest single sale recorded on 2024-12-30 with a total of **$172,316** in the North America region for Services category. This indicates potential peak seasonality effects or promotional impacts during holiday periods that may be worth exploring further.

## Product Analysis and Pe...


In [7]:
# Marketing Agent
print("🎯 Running Marketing Analyst Agent...")
marketing_agent = MarketingAnalystAgent(llm_client)
marketing_report = marketing_agent.analyze(datasets["marketing"], include_trends=True)
print("✅ Marketing analysis complete!")
print(f"⏱️ Time: {marketing_agent.get_metadata()['execution_time_seconds']}s\n")
print(marketing_report[:500] + "...")

🎯 Running Marketing Analyst Agent...


/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()
/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()
/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()


⚠️ LLM generation error: Ollama request timed out after 60s
✅ Marketing analysis complete!
⏱️ Time: 1195.15s

## Sales Analysis Insight

Based on the data provided:

1. **Revenue Trends**: Seasonal peaks in Q4 suggest holiday-driven demand.
2. **Top Performers**: Electronics and Software lead revenue.
3. **Regional Performance**: North America strongest; Asia Pacific fastest growth.
4. **Recommendations**: Increase inventory for top products before Q4.

*Fallback response — connect Ollama for AI-powered analysis.*...


In [8]:
# Customer Agent (with ML clustering)
print("👥 Running Customer Analyst Agent...")
customer_agent = CustomerAnalystAgent(llm_client)
customer_report = customer_agent.analyze(
    datasets["customers"], 
    include_clustering=True,
    n_clusters=4
)
print("✅ Customer analysis complete!")
print(f"⏱️ Time: {customer_agent.get_metadata()['execution_time_seconds']}s")
print(customer_report[:500] + "...")

👥 Running Customer Analyst Agent...


KeyboardInterrupt: 

In [8]:
# Tech Agent (with ML anomaly detection)
print("💻 Running Tech Analyst Agent...")
tech_agent = TechAnalystAgent(llm_client)
tech_report = tech_agent.analyze(
    datasets["github"],
    include_anomalies=True,
    contamination=0.1
)
print("✅ Tech analysis complete!")
print(f"⏱️ Time: {tech_agent.get_metadata()['execution_time_seconds']}s")
print(tech_report[:500] + "...")

💻 Running Tech Analyst Agent...
✅ Tech analysis complete!
⏱️ Time: 15.91s
## Language Trends Analysis

Based on the provided GitHub repository data, we can observe that Python is the most popular language with a significant number of repositories containing this as their top programming language (600 out of 600). This suggests strong community interest and possibly robust ecosystem support for projects in Python. Although not explicitly mentioned herein, it's common to see JavaScript also being widely used on GitHub due to its popularity across various domains like we...


## 8️⃣ Run Multi-Agent Orchestration

The **AgentOrchestrator** runs all agents and feeds their reports to the **Strategy Agent** for unified strategic synthesis — all powered by the fine-tuned model.

In [9]:
from agents.orchestrator import AgentOrchestrator

orchestrator = AgentOrchestrator(llm_client)

print("🚀 Starting Multi-Agent Analysis Pipeline...")
print(f"   Using model: {llm_client.model}")
print("="*60)

def progress_fn(agent_name, status):
    print(f"  🤖 {agent_name}: {status}")

results = orchestrator.run_full_analysis(
    datasets=datasets,
    include_ml=True,
    progress_callback=progress_fn
)

print("\n" + "="*60)
print("✅ Multi-Agent Analysis Complete!")
print(f"⏱️ Total time: {results['metadata']['total_execution_time']}s")
print(f"🤖 Agents run: {results['metadata']['agents_run']}")
print(f"📊 LLM Stats: {results['metadata']['llm_stats']}")

🚀 Starting Multi-Agent Analysis Pipeline...
   Using model: business-analyst
  🤖 Sales Analyst: Analyzing sales data...


/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()
/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()


  🤖 Sales Analyst: ✅ Complete
  🤖 Marketing Analyst: Analyzing marketing data...


/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()
/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()
/home/ubuntu/ml/trend_analysis.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts = df_copy.set_index(date_col)[value_col].resample(freq).sum().reset_index()


  🤖 Marketing Analyst: ✅ Complete
  🤖 Customer Analyst: Analyzing customer data...
  🤖 Customer Analyst: ✅ Complete
  🤖 Tech Analyst: Analyzing GitHub/tech data...
  🤖 Tech Analyst: ✅ Complete
  🤖 Strategy Agent: Synthesizing all insights...
  🤖 Strategy Agent: ✅ Complete
  🤖 Recommendations: Generating final recommendations...
  🤖 Recommendations: ✅ Complete

✅ Multi-Agent Analysis Complete!
⏱️ Total time: 70.98s
🤖 Agents run: ['sales', 'marketing', 'customers', 'tech']
📊 LLM Stats: {'total_calls': 11, 'total_tokens': 10213, 'total_time_seconds': 153.48, 'avg_time_per_call': 13.95, 'model': 'business-analyst'}


In [10]:
# Display Strategic Report
from IPython.display import Markdown, display

print("🧠 STRATEGIC SYNTHESIS REPORT")
print("="*60)
display(Markdown(results.get("strategic_report", "No report generated.")))

🧠 STRATEGIC SYNTHESIS REPORT


# Executive Strategic Recommendation for Cross-Functional Synergy, Risk Assessment, and Prioritized Action Items Based on Analyst Insights:

## Executive Summary:
Our organization stands at a pivotal point with stable revenue streams but faces inconsistency in sales performance. High-margin products drive growth primarily within North America while Europe and Asia Pacific present untapped opportunities for expansion, particularly if we can align our offerings to regional preferences/needs.

## Cross-Functional Insights:
Sales insights reveal variability that could impact financial stability; marketing strategies must focus on consistency across channels while Tech and Customer Analysts highlight language trends, repository quality standards, community health indicators, open source patterns, technology adoption rates, as well as correlations between CI/CD practices and success.

## Top Opportunities:
1. **North American Market Penetration** - Given the high revenue from this region with a preference for hardware products like GPU Accelerator Cards, prioritizing marketing efforts here could yield significant growth (Impact Ranked High). Priority 1 resources should be allocated to regional-focused campaigns and strategic partnerships.
    - Expected Impact: Increased sales volume in North America with a focus on high-margin products, leading to improved overall financial health without disproportionately affecting margins negatively (Priority 1).
2. **European/Asian Pacific Expansion** - With Europe and Asia Pacific regions showing potential for growth due to their interest but lower revenue per transaction compared to North America, tailored marketing campaigns could unlock these markets further (Impact Ranked Medium-High). Priority 2 resources should be allocated towards understanding regional preferences/needs.
    - Expected Impact: Enhanced sales volume across all categories without disproportionately affecting margins negatively, leading to increased revenue and market presence in these regions (Priority 2).
3. **Consistent Sales Performance** - Address the high standard deviation of revenues by implementing consistent promotional efforts throughout the year with cross-promotion between products/services where appropriate (Impact Ranked Medium). Priority 3 resources should be allocated to developing a loyalty program and regular discounts on services.
    - Expected Impact: Reduction in sales variability, leading to steadier cash flow throughout the year with less risk of significant fluctuations that can impact financial planning (Priority 3).

## Critical Risks:
1. **Inconsistent Sales Performance** – High standard deviation indicates inconsistency which could pose risks if not addressed through more consistent marketing efforts or product diversification strategies to stabilize cash flow throughout the year (Impact Ranked Medium-High). Mitigation strategy includes implementing a loyalty program and regular discounts on services, as well as cross-promotion between products/services.
    - Expected Impact: Reduction in sales variability leading to more predictable cash flow with less risk of significant fluctuations (Priority 3).
2. **Underperforming Product Categories** – USB-C Hub Electronics and Training Workshop have the lowest revenue figures, suggesting either lower market demand or less effective sales strategies which could impact overall profitability if not addressed promptly through product optimization or targeted promotions (Impact Ranked Medium). Mitigation strategy includes revisiting these products' positioning in our portfolio and exploring alternative engagement tactics.
    - Expected Impact: Improved margins for underperforming categories, leading to a more balanced product mix contributing positively towards overall profitability (Priority 2).

## Strategic Recommendations with Prioritized Action Plan:
- **Immediate Actions (0-30 days)**  
    - Launch targeted marketing campaigns in North America focusing on high-margin products, leveraging the identified preferences. Begin development of a loyalty program and regular service discounts to promote consistent sales performance across all regions with an emphasis on low performers (Priority 1).  
    - Conduct research into European/Asia Pacific markets for tailored marketing campaigns, exploring partnerships or localized offers that resonate with regional preferences/needs. This includes identifying language trends and technology adoption rates to inform product development strategies (Priority 2).
- **Short-term Initiatives (30-90 days)**  
    - Roll out loyalty programs, regular service discounts across all regions with a focus on low performers. Monitor engagement and adjust campaign tactics as needed to ensure consistent sales performance throughout the year (Priority 1).  
    - Begin developing or sourcing products that align better with regional preferences/needs in Europe and Asia Pacific, informed by initial research findings into language trends and technology adoption rates. Prioritize these initiatives based on potential impact assessments to ensure effective resource allocation (Priority 2).
- **Long-term Strategy (90+ days)**  
    - Continue refining marketing strategies for North American penetration, incorporating feedback from loyalty programs and service discounts. Focus on maintaining a strong presence in this region while exploring opportunities to expand into adjacent markets with similar preferences (Priority 1).  
    - Expand research efforts into Europe and Asia Pacific regions for deeper understanding of regional market dynamics, continuously align product offerings accordingly, ensuring that we are not just selling but also providing value in line with local demands. This includes ongoing engagement to maintain community health indicators (Priority 2).
- **Resource Allocation**  
    - Budget and talent should be allocated first towards North America for immediate marketing campaigns, followed by a significant portion into research efforts for European/Asia Pacific expansion as short-term initiatives. Resources dedicated to product development in these regions will follow based on the outcomes of initial engagements (Priority 1).  
    - Allocate resources towards developing loyalty programs and regular service discounts, ensuring that a portion is reserved for potential adjustments or enhancements as sales data becomes available. Talent should be focused first in North America with some allocated to research teams working on European/Asia Pacific market strategies (Priority 2).
- **Success Metrics**  
    - Track revenue, customer acquisition costs, and conversion rates for targeted campaigns within the high-margin product categories. Monitor loyalty program engagement metrics closely as an immediate action metric to assess impact on sales consistency (Impact Ranked High/Priority 1).  
    - Measure market research effectiveness by tracking interest in tailored products and services, with a focus on customer feedback for continuous improvement of offerings. Track technology adoption rates within Europe and Asia Pacific as an indicator to inform long-term product development strategies (Impact Ranked Medium/Priority 2).
    - Monitor the standard deviation of revenue monthly across all regions, aiming for a reduction in variability over time with regular discounts. Track customer retention and lifetime value as indicators of loyalty program success to ensure sustained engagement (Impact Ranked Medium/Priority 1).
    - Evaluate the overall impact on profit margins across all regions, prioritizing immediate actions first for a quick win in North America. Assess long-term strategy effectiveness by tracking market share growth and customer satisfaction within European/Asia Pacific markets (Impact Ranked Medium/Priority 2).
    - Set KPIs to measure the success of loyalty programs, including redemption rates for discount vouchers. Evaluate long-term strategy effectiveness by tracking market share growth and customer satisfaction within European/Asia Pacific markets (Impact Ranked Medium/Priority 2).

By following this strategic plan with immediate actions to stabilize sales performance, short-term initiatives for consistent engagement across regions, a long-term strategy focused on regional alignment of products and services, along with the recommended resource allocation and success metrics tracking, we can address our current challenges while capitalizing on growth opportunities.

In [11]:
# Display Final Recommendations
print("🎯 ACTIONABLE RECOMMENDATIONS")
print("="*60)
display(Markdown(results.get("recommendations", "No recommendations generated.")))

🎯 ACTIONABLE RECOMMENDATIONS


## Prioritized Recommendations for Strategic Enhancement Based on GitHub Analysis Insights

1. **Title**: Implement TypeScript Adoption Initiative  
    - **Priority**: High  
    - **Impact**: Significant boost in developer productivity and code quality, potentially increasing the number of high-quality projects within our ecosystem by 20% over the next year.  
    - **Effort**: Medium  
    - **Details**: Given TypeScript's growing activity on GitHub for large-scale JavaScript applications, we should invest in training and resources to encourage its adoption across projects within our ecosystem. This includes workshops, online courses, and tooling support that align with the needs of developers working primarily or exclusively with Python but are interested in expanding their skill set into TypeScript for future-proofing project development practices.

2. **Title**: Enhance Code Quality Standards  
    - **Priority**: High  
    - **Impact**: Improved overall code quality across the ecosystem, potentially increasing repository stars by 15% within two years due to better maintainability and reliability of projects.  
    - **Effort**: Medium-High  
    - **Details**: While many repositories already have good standards in place with an average code quality score above `2`, we should launch a campaign promoting best practices, including the importance of CI/CD pipelines and comprehensive documentation. This could involve creating templates or guides that are easily accessible on GitHub to assist less experienced contributors while also offering recognition for projects maintaining high standards as case studies within our community platform.

3. **Title**: Foster Continuous Community Engagement  
    - **Priority**: Medium  
    - **Impact**: Increased repository activity and contributions, leading to a more vibrant ecosystem with enhanced project sustainability over time. Expect at least 10% increase in contributor numbers across the top repositories within one year.  
    - **Effort**: Medium-Low  
    - **Details**: To address varying levels of activity, we should implement community engagement initiatives such as regular hackathons or mentorship programs that connect experienced developers with newcomers and encourage consistent contributions to repositories. Special attention will be given to high-activity areas like data science where the demand for skilled contributors is likely higher due to project complexity and scope, ensuring a steady flow of fresh ideas and maintainer support within these critical projects.

4. **Title**: Promote TypeScript Learning Resources  
    - **Priority**: Medium  
    - **Impact**: Encouraging the adoption of TypeScript in our ecosystem, potentially increasing its usage by 25% among Python-focused projects within two years. This can also lead to a diversification of skills and project approaches that may attract new contributors interested in modern JavaScript development practices.  
    - **Effort**: Low  
    - **Details**: As TypeScript shows promising engagement, we should create tailored learning resources such as curated tutorials, cheat sheets for common issues faced by Python developers transitioning to or incorporating TypeScript into their projects, and highlight successful examples of mixed-language ecosystems. These materials will be shared through our community channels with a focus on the benefits that adopting TypeScript can bring in terms of code quality assurance and scalability within JavaScript applications.

5. **Title**: Develop CI/CD Best Practices Campaign  
    - **Priority**: Low  
    - **Impact**: While not all repositories have CI/CD, promoting its adoption could lead to a 10% increase in overall repository quality scores within three years as developers become more adept at automating their workflows. This can also indirectly improve the consistency of releases and reduce time-to-market for new features or fixes across projects.  
    - **Effort**: Medium  
    - **Details**: Although CI/CD is already present in a significant portion of repositories, we should continue to advocate its importance by sharing success stories from the top 5 most anomalous records that have high star counts but lack these features. Additionally, providing resources and support for setting up basic pipelines or offering consulting services on integrating CI/CD into projects can help lower-activity areas improve their development practices without imposing significant changes to existing workflows.

By focusing our efforts strategically based on the GitHub analysis insights provided by AI Agent, we aim to enhance both individual and collective growth within this vibrant open source ecosystem while also addressing key opportunities for improvement in technology adoption and community engagement practices.

In [12]:
# Save full report to file
full_report = "# Multi-Agent Analysis Report\n\n"
for key, report in results.get("reports", {}).items():
    full_report += f"## {key.title()} Analysis\n\n{report}\n\n---\n\n"
full_report += f"## Strategic Synthesis\n\n{results.get('strategic_report', '')}\n\n---\n\n"
full_report += f"## Recommendations\n\n{results.get('recommendations', '')}\n"

with open("analysis_report.md", "w", encoding="utf-8") as f:
    f.write(full_report)

print("📄 Full report saved to: analysis_report.md")

📄 Full report saved to: analysis_report.md


## 9️⃣ Launch Streamlit Dashboard

Run the interactive dashboard for visual exploration of all analyses.  
The dashboard also uses the fine-tuned `business-analyst` model.

In [24]:
import os
import sys

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv()

import streamlit as st
import pandas as pd
import numpy as np
import plotly
import sklearn
import ollama

print("✅ Streamlit runtime imports OK")
print(f"streamlit: {st.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"plotly: {plotly.__version__}")
print(f"sklearn: {sklearn.__version__}")


✅ Streamlit runtime imports OK
streamlit: 1.29.0
pandas: 2.3.3
numpy: 1.26.4
plotly: 6.6.0
sklearn: 1.8.0


In [15]:
print("🚀 Launching Streamlit Dashboard...")
print("   URL: http://localhost:8501")
print(f"   Model: {llm_client.model}")
print("   Press Ctrl+C to stop the server")
print()

!streamlit run app/main.py --server.headless true

🚀 Launching Streamlit Dashboard...
   URL: http://localhost:8501
   Model: business-analyst
   Press Ctrl+C to stop the server




  You can now view your Streamlit app in your browser.

  Network URL: http://10.244.45.5:8501
  External URL: http://38.128.232.129:8501

/home/ubuntu/components/charts.py:70: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df_copy.groupby(pd.Grouper(key=date_col, freq="M"))[value_col].sum().reset_index()
/home/ubuntu/components/charts.py:417: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df_copy.groupby(pd.Grouper(key=x_col, freq="M"))[y_col].sum().reset_index()
/home/ubuntu/components/charts.py:417: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df_copy.groupby(pd.Grouper(key=x_col, freq="M"))[y_col].sum().reset_index()
/home/ubuntu/components/charts.py:417: FutureW

---

## 📋 System Architecture

```
┌─────────────────────────────────────────────────────────────┐
│               FINE-TUNING PIPELINE (Step 2)                 │
│  HuggingFace Datasets → QLoRA Training → Ollama Model      │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│                     DATA LAYER (Step 4)                     │
│  Any CSV → Auto-Detected (Sales/Marketing/Customer/...)    │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│               DATASET DETECTOR (Step 6)                    │
│  Column patterns → Type detection → Auto KPIs/Charts       │
└──────────────────────┬──────────────────────────────────────┘
                       │
            ┌──────────┴──────────┐
            ▼                     ▼
┌──────────────────┐  ┌──────────────────────┐
│  ML SIGNALS       │  │  LLM CORE (Step 3)   │
│  (Optional)       │  │  Fine-tuned Phi-3     │
│  • Clustering     │──│  'business-analyst'   │
│  • Anomaly Det.   │  │  • Fast English-only  │
│  • Trend Signals  │  │  • RTX A6000 GPU      │
└──────────────────┘  └────────┬─────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│                 MULTI-AGENT SYSTEM (Step 7-8)               │
│  Sales → Marketing → Customer → Tech → Strategy Agent      │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│               STREAMLIT DASHBOARD (Step 9)                  │
│  Upload Any CSV │ Auto-Viz │ AI Insights │ Multi-Agent     │
└─────────────────────────────────────────────────────────────┘
```

In [1]:
print("A")
import importlib
print("B")
import finetune.train
print("C")
importlib.reload(finetune.train)
print("D")
from finetune.train import train
print("E")


A
B
C
D
E
